# BirdCLEF 2026 — ProtoSSM
**Path B: LightProtoSSM on Perch embeddings**

Trains a sequence model (LightProtoSSM + ResidualSSM second-pass) on top of frozen
Perch embeddings of labeled soundscape sequences. Each soundscape file = 12 contiguous
5-second windows, so the sequence model can use temporal context across the whole file.

**Pipeline:**
1. Load Perch ONNX encoder
2. Parse `train_soundscapes_labels.csv`, keep only files with all 12 windows labeled
3. Embed each labeled file into a `(12, 1536)` sequence + a `(12, 234)` Perch-mapped score sequence
4. Train **LightProtoSSM**: bidirectional Mamba-style SSM + cross-attention + learned class prototypes + site/hour metadata
5. Validate via GroupKFold (group = filename) — competition macro-AUC on held-out files
6. Train **ResidualSSM**: small second-pass corrector that predicts `(y_true − sigmoid(first_pass))`
7. Save bundle to `notebooks/protossm/protossm_run/` for the Kaggle inference notebook

**What this replaces vs. v2.7/v2.8:** the focal-clip training cache, the residual MLP head, and Stage 8 pseudo-labeling are dropped — they're not needed for ProtoSSM and would slow the notebook down. They live on in your existing `perch_baseline-v2_7-and-v2_8` notebook if you want to come back to them.


## Stage 0 — Install dependencies

In [1]:
%pip install -q onnxruntime kagglehub librosa scikit-learn tqdm soundfile
%pip install -q torch

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Stage 0b — Download Perch ONNX model (one-time)

In [2]:
import kagglehub
from pathlib import Path

_perch_onnx_dir = Path(kagglehub.dataset_download('rishikeshjani/perch-onnx-for-birdclef-2026'))
print('Downloaded to:', _perch_onnx_dir)
print('\nFiles in dataset:')
for f in sorted(_perch_onnx_dir.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(_perch_onnx_dir)}  ({f.stat().st_size/1e6:.1f} MB)')

_onnx_files = list(_perch_onnx_dir.rglob('*.onnx'))
if not _onnx_files:
    raise FileNotFoundError('No .onnx file found in dataset')
PERCH_ONNX_PATH = _onnx_files[0]
print(f'\nUsing ONNX file: {PERCH_ONNX_PATH}')

Downloaded to: /Users/enricozafiris/.cache/kagglehub/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/versions/2

Files in dataset:
  labels.csv  (0.3 MB)
  onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl  (17.2 MB)
  perch_v2.onnx  (409.1 MB)

Using ONNX file: /Users/enricozafiris/.cache/kagglehub/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/versions/2/perch_v2.onnx


## Stage 1 — Imports, paths, constants

In [3]:
from __future__ import annotations
import gc, json, re, time, warnings
from pathlib import Path

import librosa
import numpy as np
import onnxruntime as ort
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

# ── Absolute paths (your local M1 setup) ─────────────────────────────────
PROJECT_ROOT          = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project')
DATA_DIR              = PROJECT_ROOT / 'data'
TRAIN_AUDIO_DIR       = DATA_DIR / 'train_audio'
TRAIN_SOUNDSCAPES_DIR = DATA_DIR / 'train_soundscapes'
TRAIN_CSV             = DATA_DIR / 'train.csv'
LABELS_CSV            = DATA_DIR / 'train_soundscapes_labels.csv'
SAMPLE_SUB_CSV        = DATA_DIR / 'sample_submission.csv'
TAXONOMY_CSV          = DATA_DIR / 'taxonomy.csv'

# Cache for Perch sequence embeddings (one entry per labeled soundscape file)
CACHE_DIR = PROJECT_ROOT / 'notebooks' / 'protossm' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NPZ_SEQ   = CACHE_DIR / 'protossm_sequences.npz'

# Output bundle for the Kaggle inference notebook
AUTOSAVE_ROOT     = PROJECT_ROOT / 'notebooks' / 'protossm'
AUTOSAVE_RUN_NAME = 'protossm_run'
AUTOSAVE_DIR      = AUTOSAVE_ROOT / AUTOSAVE_RUN_NAME
AUTOSAVE_DIR.mkdir(parents=True, exist_ok=True)

# Perch label vocabulary (downloaded by kagglehub.model_download earlier)
PERCH_LABELS_CSV = Path(
    '/Users/enricozafiris/.cache/kagglehub/models/'
    'google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/1/'
    'assets/labels.csv'
)

# ── Audio constants ──────────────────────────────────────────────────────
SR_LOAD       = 32_000
PERCH_SR      = 32_000
WINDOW_SEC    = 5.0
N_WINDOWS     = 12                              # 60s soundscape ÷ 5s = 12 windows
EMB_DIM       = 1536
PERCH_SAMPLES = int(WINDOW_SEC * PERCH_SR)      # 160_000
FILE_SAMPLES  = N_WINDOWS * PERCH_SAMPLES        # 1_920_000

RNG = np.random.default_rng(42)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('AUTOSAVE_DIR:', AUTOSAVE_DIR)
print('PERCH_ONNX_PATH:', PERCH_ONNX_PATH)
print('PERCH_LABELS_CSV exists:', PERCH_LABELS_CSV.is_file())
if not PERCH_LABELS_CSV.is_file():
    print('  ⚠ Run: kagglehub.model_download("google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu")')

PROJECT_ROOT: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project
AUTOSAVE_DIR: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/protossm/protossm_run
PERCH_ONNX_PATH: /Users/enricozafiris/.cache/kagglehub/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/versions/2/perch_v2.onnx
PERCH_LABELS_CSV exists: True


## Stage 2 — Load ONNX Perch encoder

In [4]:
print('Loading Perch ONNX session...', flush=True)
t0 = time.time()
_so = ort.SessionOptions()
_so.intra_op_num_threads = 4
_so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
_perch_session = ort.InferenceSession(
    str(PERCH_ONNX_PATH), sess_options=_so, providers=['CPUExecutionProvider'],
)
print(f'  Loaded in {time.time()-t0:.2f}s')

print('\nModel inputs:')
for inp in _perch_session.get_inputs():
    print(f'  name={inp.name}  shape={inp.shape}  dtype={inp.type}')
print('\nModel outputs:')
for out in _perch_session.get_outputs():
    print(f'  name={out.name}  shape={out.shape}  dtype={out.type}')

_PERCH_INPUT_NAME = _perch_session.get_inputs()[0].name
_PERCH_OUTPUT_NAMES = [o.name for o in _perch_session.get_outputs()]

# Smoke test: identify embedding (1536-d) and logits (~14795-d) outputs
_dummy_in = np.zeros((1, PERCH_SAMPLES), dtype=np.float32)
_outs = _perch_session.run(None, {_PERCH_INPUT_NAME: _dummy_in})
_EMB_IDX, _LOGIT_IDX = None, None
for i, arr in enumerate(_outs):
    print(f'  output[{i}] {_PERCH_OUTPUT_NAMES[i]}: shape={arr.shape}')
    if arr.ndim == 2 and arr.shape[-1] == EMB_DIM:
        _EMB_IDX = i
    elif arr.ndim == 2 and arr.shape[-1] > 5000:
        _LOGIT_IDX = i
print(f'EMB_IDX={_EMB_IDX}  LOGIT_IDX={_LOGIT_IDX}')
assert _EMB_IDX is not None and _LOGIT_IDX is not None

Loading Perch ONNX session...
  Loaded in 1.71s

Model inputs:
  name=inputs  shape=['batch', 160000]  dtype=tensor(float)

Model outputs:
  name=embedding  shape=['batch', 1536]  dtype=tensor(float)
  name=spatial_embedding  shape=['batch', 16, 4, 1536]  dtype=tensor(float)
  name=spectrogram  shape=['batch', 500, 128]  dtype=tensor(float)
  name=label  shape=['batch', 14795]  dtype=tensor(float)
  output[0] embedding: shape=(1, 1536)
  output[1] spatial_embedding: shape=(1, 16, 4, 1536)
  output[2] spectrogram: shape=(1, 500, 128)
  output[3] label: shape=(1, 14795)
EMB_IDX=0  LOGIT_IDX=3


In [5]:
def perch_embed_batch(waveforms_5s):
    """Embed a batch of 5-second waveforms (32 kHz) through Perch.
    Returns:
        embeddings: (B, 1536) float32
        logits:     (B, ~14795) float32
    """
    if isinstance(waveforms_5s, list):
        batch = np.stack(waveforms_5s, axis=0).astype(np.float32)
    else:
        batch = np.asarray(waveforms_5s, dtype=np.float32)
    if batch.ndim == 1:
        batch = batch[None, :]
    if batch.shape[1] != PERCH_SAMPLES:
        fixed = np.zeros((batch.shape[0], PERCH_SAMPLES), dtype=np.float32)
        for i, row in enumerate(batch):
            n = min(len(row), PERCH_SAMPLES)
            fixed[i, :n] = row[:n]
        batch = fixed
    outs = _perch_session.run(None, {_PERCH_INPUT_NAME: batch})
    return outs[_EMB_IDX].astype(np.float32), outs[_LOGIT_IDX].astype(np.float32)

print('Perch embed helper ready.')

Perch embed helper ready.


## Stage 3 — Species, taxonomy, and Perch-logit mapping

Same direct-mapping + genus-proxy logic as v2.8. We need this so that the Perch
**zero-shot scores** (234-dim) become the second input to ProtoSSM (alongside the
1536-d embedding). The reference notebook calls this `perch_logits` in the model
forward pass.

In [6]:
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
species_cols = [c for c in sample_sub.columns if c != 'row_id']
species_to_idx = {s: i for i, s in enumerate(species_cols)}
n_species = len(species_cols)
print('n_species:', n_species)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)

# ── Direct logit mapping ────────────────────────────────────────────────
perch_labels = pd.read_csv(PERCH_LABELS_CSV)
sci_col_candidates = [c for c in perch_labels.columns if any(
    k in c.lower() for k in ['sci', 'name', 'inat', 'fsd']
)]
sci_col = sci_col_candidates[0] if sci_col_candidates else perch_labels.columns[0]
print(f"Using column '{sci_col}' for scientific names")

perch_labels = perch_labels.reset_index().rename(columns={'index': 'perch_idx'})
perch_sci_to_idx = dict(zip(
    perch_labels[sci_col].astype(str).str.strip(),
    perch_labels['perch_idx'].astype(int),
))
NO_LABEL = len(perch_labels)

tax_sci   = taxonomy_df.set_index('primary_label')['scientific_name'].to_dict()
tax_class = taxonomy_df.set_index('primary_label')['class_name'].to_dict()

BC_INDICES = []
for sp in species_cols:
    sci = str(tax_sci.get(sp, '')).strip()
    BC_INDICES.append(perch_sci_to_idx.get(sci, NO_LABEL))
BC_INDICES = np.array(BC_INDICES, dtype=np.int32)

MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)

# ── Genus proxy for Aves / Amphibia / Insecta ──────────────────────────
PROXY_TAXA = {'Aves', 'Amphibia', 'Insecta'}
proxy_map = {}
for sp_idx in UNMAPPED_POS:
    sp  = species_cols[int(sp_idx)]
    cls = tax_class.get(sp, '')
    if cls not in PROXY_TAXA:
        continue
    sci = str(tax_sci.get(sp, '')).strip()
    genus = sci.split()[0] if sci else ''
    if not genus:
        continue
    pat = re.compile(rf'^{re.escape(genus)}\s')
    hits = [pidx for psci, pidx in perch_sci_to_idx.items() if pat.match(psci)]
    if hits:
        proxy_map[int(sp_idx)] = hits

print(f'Direct mapped: {MAPPED_MASK.sum()} / {n_species}')
print(f'Genus proxy:   {len(proxy_map)} / {len(UNMAPPED_POS)} unmapped')

def apply_perch_mapping(logits):
    """Convert raw Perch logits (B, vocab) -> BirdCLEF species scores (B, 234)."""
    B = logits.shape[0]
    out = np.broadcast_to(
        logits.mean(axis=1, keepdims=True), (B, n_species)
    ).astype(np.float32).copy()
    out[:, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
    for sp_idx, bc_idxs in proxy_map.items():
        out[:, sp_idx] = logits[:, bc_idxs].mean(axis=1)
    return out

n_species: 234
Using column 'inat2024_fsd50k' for scientific names
Direct mapped: 203 / 234
Genus proxy:   3 / 31 unmapped


## Stage 4 — Parse soundscape labels & filter to fully-labeled files

Each soundscape file is 60s = 12 contiguous 5-second windows. ProtoSSM needs
**complete sequences** of length 12, so we keep only files where all 12 windows
appear in `train_soundscapes_labels.csv`. Following the reference notebook this
gives ~59 files / ~708 windows on the public competition data — that's small but
each window is densely labeled, and the sequence structure is what gives ProtoSSM
its lift over the flat MLP head.

We also extract `site` and `hour` from the filename for the metadata embeddings
the model uses.

In [7]:
FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': 'unknown', 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(';'):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

# Build per-window label table
lab_raw = pd.read_csv(LABELS_CSV)
sc = (lab_raw
      .groupby(['filename', 'start', 'end'])['primary_label']
      .apply(union_labels)
      .reset_index(name='label_list'))

sc['end_sec'] = pd.to_timedelta(sc['end']).dt.total_seconds().astype(int)
sc['row_id'] = sc['filename'].str.replace('.ogg', '', regex=False) + '_' + sc['end_sec'].astype(str)

_meta = sc['filename'].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), n_species), dtype=np.float32)
for i, lbls in enumerate(sc['label_list']):
    for lbl in lbls:
        if lbl in species_to_idx:
            Y_SC[i, species_to_idx[lbl]] = 1.0

# Keep only files with all 12 windows present in the labels CSV
windows_per_file = sc.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc['fully_labeled'] = sc['filename'].isin(full_files)

# Also restrict to files that actually exist on disk
existing = []
for fn in full_files:
    if (TRAIN_SOUNDSCAPES_DIR / fn).is_file():
        existing.append(fn)
full_files = existing
print(f'Fully-labeled files on disk: {len(full_files)}')

# Sort windows per file by end_sec so the sequence is in temporal order
full_rows = (sc[sc['filename'].isin(full_files)]
             .sort_values(['filename', 'end_sec'])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows['index'].to_numpy()]

n_active = int((Y_FULL.sum(0) > 0).sum())
print(f'Full-file windows: {len(full_rows)}  | Active classes: {n_active} / {n_species}')

Fully-labeled files on disk: 59
Full-file windows: 708  | Active classes: 71 / 234


## Stage 5 — Embed full soundscape sequences with Perch

For each fully-labeled soundscape we read the whole 60s file, slice into the 12
contiguous 5-second windows, and run Perch once per file (batched). Output:

- `EMB_SEQ`  : `(n_files, 12, 1536)`  — Perch embeddings
- `SCR_SEQ`  : `(n_files, 12, 234)`   — Perch zero-shot scores (mapped + genus proxy)
- `Y_SEQ`    : `(n_files, 12, 234)`   — multi-hot ground truth labels
- `META_SEQ` : `(n_files,)` rows      — filename, site, hour

In [8]:
FORCE_REBUILD_SEQ = False

def read_60s(path):
    """Load a 60-second audio file as float32 mono array, padded/cropped to FILE_SAMPLES."""
    try:
        y, sr = sf.read(str(path), dtype='float32', always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr != SR_LOAD:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR_LOAD)
    except Exception:
        y, _ = librosa.load(str(path), sr=SR_LOAD, mono=True)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y.astype(np.float32)


if not FORCE_REBUILD_SEQ and NPZ_SEQ.is_file():
    d = np.load(NPZ_SEQ, allow_pickle=True)
    EMB_SEQ  = d['EMB_SEQ'].astype(np.float32)
    SCR_SEQ  = d['SCR_SEQ'].astype(np.float32)
    Y_SEQ    = d['Y_SEQ'].astype(np.float32)
    seq_files = list(d['files'])
    seq_sites = list(d['sites'])
    seq_hours = d['hours'].astype(np.int64)
    print(f'Loaded sequence cache: EMB={EMB_SEQ.shape}  SCR={SCR_SEQ.shape}  Y={Y_SEQ.shape}')
else:
    print('Building sequence cache from labeled soundscapes...')
    seq_files, seq_sites, seq_hours = [], [], []
    EMB_LIST, SCR_LIST, Y_LIST = [], [], []

    BATCH_FILES = 4   # 4 files * 12 windows = 48 windows per ONNX call

    full_rows_grouped = full_rows.groupby('filename', sort=False)
    file_order = [fn for fn in full_files]   # already sorted

    t0 = time.time()
    for start in tqdm(range(0, len(file_order), BATCH_FILES), desc='Seq embed'):
        chunk_files = file_order[start:start+BATCH_FILES]
        batch_audio = []
        valid_files = []
        for fn in chunk_files:
            try:
                y = read_60s(TRAIN_SOUNDSCAPES_DIR / fn)
            except Exception as e:
                print(f'  ⚠  skip {fn}: {e}')
                continue
            batch_audio.append(y.reshape(N_WINDOWS, PERCH_SAMPLES))
            valid_files.append(fn)
        if not valid_files:
            continue

        x = np.concatenate(batch_audio, axis=0)        # (n_valid * 12, 160000)
        embs, logits = perch_embed_batch(x)
        scores = apply_perch_mapping(logits)

        for i, fn in enumerate(valid_files):
            s = i * N_WINDOWS
            e = s + N_WINDOWS
            EMB_LIST.append(embs[s:e])                  # (12, 1536)
            SCR_LIST.append(scores[s:e])                # (12, 234)

            grp = full_rows_grouped.get_group(fn).sort_values('end_sec')
            yseq = Y_SC[grp['index'].to_numpy()]        # (12, 234)
            Y_LIST.append(yseq)

            seq_files.append(fn)
            meta = parse_fname(fn)
            seq_sites.append(meta['site'])
            seq_hours.append(meta['hour_utc'])

    EMB_SEQ   = np.stack(EMB_LIST).astype(np.float32)   # (n_files, 12, 1536)
    SCR_SEQ   = np.stack(SCR_LIST).astype(np.float32)   # (n_files, 12, 234)
    Y_SEQ     = np.stack(Y_LIST).astype(np.float32)     # (n_files, 12, 234)
    seq_hours = np.array(seq_hours, dtype=np.int64)

    np.savez_compressed(
        NPZ_SEQ,
        EMB_SEQ=EMB_SEQ, SCR_SEQ=SCR_SEQ, Y_SEQ=Y_SEQ,
        files=np.array(seq_files, dtype=object),
        sites=np.array(seq_sites, dtype=object),
        hours=seq_hours,
    )
    print(f'\nDone in {time.time()-t0:.1f}s | saved {NPZ_SEQ}')
    print(f'EMB={EMB_SEQ.shape}  SCR={SCR_SEQ.shape}  Y={Y_SEQ.shape}')

n_files = EMB_SEQ.shape[0]
print(f'\nSequence cache: {n_files} files x {N_WINDOWS} windows')

Loaded sequence cache: EMB=(59, 12, 1536)  SCR=(59, 12, 234)  Y=(59, 12, 234)

Sequence cache: 59 files x 12 windows


## Stage 6 — Site / hour metadata indices

The model has small embedding tables for site and hour. We assign integer ids
once here so they line up between train and validation folds.

In [9]:
N_SITES_CAP = 20

unique_sites = sorted(set(seq_sites))
site2i = {s: i + 1 for i, s in enumerate(unique_sites)}   # 0 reserved for unknown
site_ids = np.array(
    [min(site2i.get(s, 0), N_SITES_CAP - 1) for s in seq_sites],
    dtype=np.int64,
)
hour_ids = np.array([h % 24 if h >= 0 else 0 for h in seq_hours], dtype=np.int64)
print(f'Sites: {len(unique_sites)}  (capped at {N_SITES_CAP})')
print(f'Hours seen: {sorted(set(int(h) for h in hour_ids))}')

Sites: 8  (capped at 20)
Hours seen: [0, 1, 2, 3, 4, 6, 7, 18, 19, 20, 21, 22, 23]


## Stage 7 — LightProtoSSM model

Faithful port of the reference architecture:

- **SelectiveSSM** layer (Mamba-inspired): selective scan with input-dependent dt/B/C
- **LightProtoSSM**: input projection -> +pos -> +(site, hour) -> 2× bidirectional SSM blocks with merge + LayerNorm
- 2× cross-attention layers so each window attends to the other 11
- Learned **prototype vector per class**, similarity = `cosine * temperature + bias`
- **Fusion gate** per class: blends prototype-similarity logit with the raw Perch zero-shot logit

In [10]:
class SelectiveSSM(nn.Module):
    """Simplified selective state space model layer (Mamba-inspired)."""
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d  = nn.Conv1d(d_model, d_model, d_conv,
                                  padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D     = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A  = -torch.exp(self.A_log)
        B  = self.B_proj(x_conv)
        C  = self.C_proj(x_conv)
        h  = torch.zeros(B_sz, D, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h  = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    """Bidirectional SSM + cross-attention + learned prototypes over Perch embeddings."""
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=12, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(2)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop      = nn.Dropout(dropout)

        if use_cross_attn:
            self.cross_attn = nn.ModuleList([
                nn.MultiheadAttention(d_model, num_heads=cross_attn_heads,
                                      dropout=dropout, batch_first=True)
                for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])

        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        """Mean-of-positives initialization in projected space."""
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]
        for i, (fwd, bwd, merge, norm) in enumerate(zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            if self.use_cross_attn:
                attn_out, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](h + attn_out)
        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = (torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp)
               + self.class_bias[None, None, :])
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1 - alpha) * perch_logits
        return sim

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print('LightProtoSSM defined.')

LightProtoSSM defined.


## Stage 8 — Out-of-fold validation (GroupKFold by file)

Critical sanity check: the model can easily overfit 59 files. We train K folds
where each fold holds out a group of files, predict on the held-out files, and
report **competition macro-AUC** on the concatenated OOF predictions.

If macro-AUC here is not at least ~0.80 you should not trust the final model.

Also gives us reliable training first-pass predictions to feed into ResidualSSM
in the next stage.

In [ ]:
device = torch.device('cpu')   # CPU on M1 — MPS has shape quirks for SSM
print('Using device:', device)

def macro_auc(y_true, y_score):
    """Competition metric: macro ROC-AUC, skipping classes with no positives."""
    keep = y_true.sum(axis=0) > 0
    if not np.any(keep):
        return float('nan')
    return float(roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro'))


def train_one_fold(emb_tr, scr_tr, y_tr, site_tr, hour_tr,
                   emb_va, scr_va, y_va, site_va, hour_va,
                   n_epochs=40, lr=8e-4, weight_decay=1e-3,
                   patience=8, distill_weight=0.15, swa_frac=0.65, swa_lr=4e-4,
                   verbose=False):
    emb_t  = torch.tensor(emb_tr,  dtype=torch.float32, device=device)
    scr_t  = torch.tensor(scr_tr,  dtype=torch.float32, device=device)
    lab_t  = torch.tensor(y_tr,    dtype=torch.float32, device=device)
    sti_t  = torch.tensor(site_tr, dtype=torch.long,    device=device)
    hri_t  = torch.tensor(hour_tr, dtype=torch.long,    device=device)
    emb_v  = torch.tensor(emb_va,  dtype=torch.float32, device=device)
    scr_v  = torch.tensor(scr_va,  dtype=torch.float32, device=device)
    sti_v  = torch.tensor(site_va, dtype=torch.long,    device=device)
    hri_v  = torch.tensor(hour_va, dtype=torch.long,    device=device)

    model = LightProtoSSM(
        d_input=EMB_DIM, n_classes=n_species, n_windows=N_WINDOWS,
        n_sites=N_SITES_CAP, use_cross_attn=True, cross_attn_heads=2,
    ).to(device)

    # Init prototypes from positive-mean embeddings (flatten train sequences)
    model.init_prototypes(
        emb_t.reshape(-1, EMB_DIM),
        lab_t.reshape(-1, n_species),
    )

    pos_cnt = lab_t.sum(dim=(0, 1))
    total   = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy='cos',
    )
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(n_epochs * swa_frac)
    swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=swa_lr)

    best_loss, best_state, wait = float('inf'), None, 0
    last_ep = 0
    for ep in range(n_epochs):
        last_ep = ep
        model.train()
        out  = model(emb_t, scr_t, site_ids=sti_t, hours=hri_t)
        loss = (F.binary_cross_entropy_with_logits(
                    out, lab_t, pos_weight=pos_weight[None, None, :])
                + distill_weight * F.mse_loss(out, scr_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if ep >= swa_start:
            swa_model.update_parameters(model); swa_sched.step()
        else:
            sched.step()

        if loss.item() < best_loss - 1e-4:
            best_loss = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if verbose and ep % 5 == 0:
            print(f'    ep {ep:2d}  loss={loss.item():.4f}  best={best_loss:.4f}')
        if wait >= patience:
            break

    if last_ep >= swa_start:
        # AveragedModel wraps keys as "module.*" — unwrap for clean forward + save
        model = swa_model.module
    else:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        pred_va = model(emb_v, scr_v, site_ids=sti_v, hours=hri_v).cpu().numpy()
    return model, pred_va, best_loss


# ── Run GroupKFold (group = filename) ──
N_SPLITS = min(5, n_files)
gkf = GroupKFold(n_splits=N_SPLITS)
groups = np.array(seq_files)

oof_pred = np.zeros_like(Y_SEQ)
fold_losses = []

t_total = time.time()
for fold, (tr_idx, va_idx) in enumerate(gkf.split(EMB_SEQ, groups=groups)):
    t0 = time.time()
    print(f'\n── Fold {fold+1}/{N_SPLITS}  train={len(tr_idx)}  val={len(va_idx)} ──')
    _, pred_va, best_loss = train_one_fold(
        EMB_SEQ[tr_idx], SCR_SEQ[tr_idx], Y_SEQ[tr_idx],
        site_ids[tr_idx], hour_ids[tr_idx],
        EMB_SEQ[va_idx], SCR_SEQ[va_idx], Y_SEQ[va_idx],
        site_ids[va_idx], hour_ids[va_idx],
        n_epochs=40, verbose=False,
    )
    oof_pred[va_idx] = pred_va
    fold_losses.append(best_loss)
    print(f'  fold {fold+1} done in {time.time()-t0:.1f}s  best_train_loss={best_loss:.4f}')

print(f'\nTotal OOF time: {time.time()-t_total:.1f}s')

# Convert logits -> probabilities and report metrics
oof_probs = 1.0 / (1.0 + np.exp(-np.clip(oof_pred, -30, 30)))
oof_flat   = oof_probs.reshape(-1, n_species)
y_flat     = Y_SEQ.reshape(-1, n_species)
scr_flat   = SCR_SEQ.reshape(-1, n_species)
perch_probs = 1.0 / (1.0 + np.exp(-scr_flat))

auc_proto = macro_auc(y_flat, oof_flat)
auc_perch = macro_auc(y_flat, perch_probs)
print('=' * 60)
print(f'  Direct Perch (zero-shot):  macro AUC = {auc_perch:.4f}')
print(f'  ProtoSSM OOF:              macro AUC = {auc_proto:.4f}')
print(f'  Δ vs Perch:                {auc_proto - auc_perch:+.4f}')
print('=' * 60)

Using device: cpu

── Fold 1/5  train=47  val=12 ──
  fold 1 done in 42.0s  best_train_loss=0.9764

── Fold 2/5  train=47  val=12 ──
  fold 2 done in 41.0s  best_train_loss=0.9864

── Fold 3/5  train=47  val=12 ──
  fold 3 done in 42.6s  best_train_loss=0.9755

── Fold 4/5  train=47  val=12 ──
  fold 4 done in 38.7s  best_train_loss=0.9629

── Fold 5/5  train=48  val=11 ──
  fold 5 done in 38.8s  best_train_loss=0.9687

Total OOF time: 203.2s
  Direct Perch (zero-shot):  macro AUC = 0.6646
  ProtoSSM OOF:              macro AUC = 0.8132
  Δ vs Perch:                +0.1486


## Stage 9 — Train final ProtoSSM on **all** files

Now that we trust OOF, train the production model on every file. This is what
the inference notebook will load.

In [12]:
print('Training final ProtoSSM on all files...')

emb_all  = torch.tensor(EMB_SEQ, dtype=torch.float32, device=device)
scr_all  = torch.tensor(SCR_SEQ, dtype=torch.float32, device=device)
lab_all  = torch.tensor(Y_SEQ,   dtype=torch.float32, device=device)
sti_all  = torch.tensor(site_ids, dtype=torch.long,   device=device)
hri_all  = torch.tensor(hour_ids, dtype=torch.long,   device=device)

proto_model = LightProtoSSM(
    d_input=EMB_DIM, n_classes=n_species, n_windows=N_WINDOWS,
    n_sites=N_SITES_CAP, use_cross_attn=True, cross_attn_heads=2,
).to(device)

proto_model.init_prototypes(
    emb_all.reshape(-1, EMB_DIM),
    lab_all.reshape(-1, n_species),
)
print(f'ProtoSSM params: {proto_model.count_parameters():,}')

pos_cnt = lab_all.sum(dim=(0, 1))
total   = lab_all.shape[0] * lab_all.shape[1]
pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

N_EPOCHS_FINAL = 40
opt   = torch.optim.AdamW(proto_model.parameters(), lr=8e-4, weight_decay=1e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=8e-4, epochs=N_EPOCHS_FINAL, steps_per_epoch=1,
    pct_start=0.1, anneal_strategy='cos',
)
swa_model = torch.optim.swa_utils.AveragedModel(proto_model)
swa_start = int(N_EPOCHS_FINAL * 0.65)
swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=4e-4)

best_loss, best_state, wait = float('inf'), None, 0
PATIENCE = 10

t0 = time.time()
for ep in range(N_EPOCHS_FINAL):
    proto_model.train()
    out  = proto_model(emb_all, scr_all, site_ids=sti_all, hours=hri_all)
    loss = (F.binary_cross_entropy_with_logits(
                out, lab_all, pos_weight=pos_weight[None, None, :])
            + 0.15 * F.mse_loss(out, scr_all))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(proto_model.parameters(), 1.0)
    opt.step()

    if ep >= swa_start:
        swa_model.update_parameters(proto_model); swa_sched.step()
    else:
        sched.step()

    if loss.item() < best_loss - 1e-4:
        best_loss  = loss.item()
        best_state = {k: v.clone() for k, v in proto_model.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if ep % 5 == 0 or ep == N_EPOCHS_FINAL - 1:
        print(f'  ep {ep:2d}  loss={loss.item():.4f}  best={best_loss:.4f}')
    if wait >= PATIENCE:
        print(f'  early stop at ep {ep}')
        break

# Pick which weights to use — unwrap AveragedModel so forward() works normally
if ep >= swa_start:
    proto_final = swa_model.module   # strips "module." prefix from state dict
    print(f'Using SWA-averaged weights (ep>={swa_start})')
else:
    proto_model.load_state_dict(best_state)
    proto_final = proto_model
    print(f'Using best-loss weights (loss={best_loss:.4f})')

proto_final.eval()
print(f'Final ProtoSSM trained in {time.time()-t0:.1f}s')

# Compute training first-pass predictions for ResidualSSM
with torch.no_grad():
    first_pass_logits = proto_final(emb_all, scr_all, site_ids=sti_all, hours=hri_all).cpu().numpy()
print(f'Training first-pass logits: {first_pass_logits.shape}')

Training final ProtoSSM on all files...
ProtoSSM params: 723,093
  ep  0  loss=1.2178  best=1.2178
  ep  5  loss=1.1091  best=1.1091
  ep 10  loss=1.0359  best=1.0359
  ep 15  loss=1.0057  best=1.0057
  ep 20  loss=0.9943  best=0.9943
  ep 25  loss=0.9888  best=0.9888
  ep 30  loss=0.9854  best=0.9854
  ep 35  loss=0.9819  best=0.9819
  ep 39  loss=0.9789  best=0.9789
Using SWA-averaged weights (ep>=26)
Final ProtoSSM trained in 47.7s
Training first-pass logits: (59, 12, 234)


## Stage 10 — ResidualSSM (second-pass corrector)

Small bidirectional SSM that takes `[embedding, first_pass_logits]` and predicts
the **residual** `y_true − sigmoid(first_pass)`. The output head is initialized
to zero so corrections start small. At inference time we add `correction_weight
× residual_pred` to the first pass.

In [13]:
class ResidualSSM(nn.Module):
    """Lightweight second-pass: predicts (y_true - sigmoid(first_pass))."""
    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd   = SelectiveSSM(d_model, d_state)
        self.ssm_bwd   = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm  = nn.LayerNorm(d_model)
        self.ssm_drop  = nn.Dropout(dropout)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass], dim=-1)
        h = self.input_proj(x) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                 self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta.unsqueeze(1)
        res = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h   = self.ssm_drop(self.ssm_merge(torch.cat([h_f, h_b], dim=-1)))
        h   = self.ssm_norm(h + res)
        return self.output_head(h)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


# Train: split files 85/15 train/val
n_val = max(1, int(n_files * 0.15))
perm  = np.random.default_rng(42).permutation(n_files)
val_i, train_i = perm[:n_val], perm[n_val:]
print(f'ResidualSSM split: train={len(train_i)}  val={len(val_i)}')

residuals = Y_SEQ - sigmoid_np(first_pass_logits)

emb_t  = torch.tensor(EMB_SEQ,            dtype=torch.float32, device=device)
fp_t   = torch.tensor(first_pass_logits,  dtype=torch.float32, device=device)
res_t  = torch.tensor(residuals,          dtype=torch.float32, device=device)
sti_t  = torch.tensor(site_ids,           dtype=torch.long,    device=device)
hri_t  = torch.tensor(hour_ids,           dtype=torch.long,    device=device)

res_model = ResidualSSM(
    d_input=EMB_DIM, d_scores=n_species, n_classes=n_species,
    n_windows=N_WINDOWS, n_sites=N_SITES_CAP,
).to(device)
print(f'ResidualSSM params: {res_model.count_parameters():,}')

opt   = torch.optim.AdamW(res_model.parameters(), lr=8e-4, weight_decay=1e-3)
N_EP  = 20
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=8e-4, epochs=N_EP, steps_per_epoch=1,
    pct_start=0.1, anneal_strategy='cos',
)

best_val, best_state, wait = float('inf'), None, 0
PATIENCE_RES = 6

train_idx_t = torch.tensor(train_i, dtype=torch.long, device=device)
val_idx_t   = torch.tensor(val_i,   dtype=torch.long, device=device)

for ep in range(N_EP):
    res_model.train()
    corr = res_model(
        emb_t[train_idx_t], fp_t[train_idx_t],
        site_ids=sti_t[train_idx_t], hours=hri_t[train_idx_t],
    )
    loss = F.mse_loss(corr, res_t[train_idx_t])
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(res_model.parameters(), 1.0)
    opt.step(); sched.step()

    res_model.eval()
    with torch.no_grad():
        val_loss = F.mse_loss(
            res_model(emb_t[val_idx_t], fp_t[val_idx_t],
                      site_ids=sti_t[val_idx_t], hours=hri_t[val_idx_t]),
            res_t[val_idx_t],
        ).item()

    if val_loss < best_val - 1e-6:
        best_val, best_state, wait = val_loss, {k: v.clone() for k, v in res_model.state_dict().items()}, 0
    else:
        wait += 1
    if ep % 3 == 0 or ep == N_EP - 1:
        print(f'  ep {ep:2d}  train={loss.item():.5f}  val={val_loss:.5f}  best_val={best_val:.5f}')
    if wait >= PATIENCE_RES:
        print(f'  early stop at ep {ep}')
        break

res_model.load_state_dict(best_state)
res_model.eval()
print(f'ResidualSSM done — best val MSE={best_val:.6f}')

CORRECTION_WEIGHT = 0.30
print(f'Correction weight to use at inference: {CORRECTION_WEIGHT}')

ResidualSSM split: train=51  val=8
ResidualSSM params: 176,010
  ep  0  train=0.20470  val=0.19842  best_val=0.19842
  ep  3  train=0.16125  val=0.13068  best_val=0.13068
  ep  6  train=0.09547  val=0.07427  best_val=0.07427
  ep  9  train=0.05641  val=0.04575  best_val=0.04575
  ep 12  train=0.03886  val=0.03452  best_val=0.03452
  ep 15  train=0.03244  val=0.03092  best_val=0.03092
  ep 18  train=0.03097  val=0.03029  best_val=0.03029
  ep 19  train=0.03091  val=0.03029  best_val=0.03029
ResidualSSM done — best val MSE=0.030289
Correction weight to use at inference: 0.3


## Stage 11 — Final OOF metric with ResidualSSM correction

Apply the correction to OOF predictions and confirm the metric still holds up.
ResidualSSM was trained on the **final** model's predictions, not on OOF, so this
will be optimistic on the training files. The honest OOF number from Stage 8 is
the one to trust.

In [14]:
# Apply the residual correction on top of training first-pass predictions
with torch.no_grad():
    res_corr = res_model(emb_t, fp_t, site_ids=sti_t, hours=hri_t).cpu().numpy()

corrected_logits = first_pass_logits + CORRECTION_WEIGHT * res_corr
corrected_probs  = sigmoid_np(corrected_logits).reshape(-1, n_species)

auc_corrected = macro_auc(y_flat, corrected_probs)
print('=' * 60)
print(f'  Direct Perch:                   {auc_perch:.4f}')
print(f'  ProtoSSM OOF (honest):          {auc_proto:.4f}')
print(f'  ProtoSSM training (overfit):    {macro_auc(y_flat, sigmoid_np(first_pass_logits).reshape(-1, n_species)):.4f}')
print(f'  + ResidualSSM (training, optimistic): {auc_corrected:.4f}')
print('=' * 60)
print('Note: only the OOF number ({:.4f}) reflects generalization.'.format(auc_proto))

  Direct Perch:                   0.6646
  ProtoSSM OOF (honest):          0.8132
  ProtoSSM training (overfit):    0.9567
  + ResidualSSM (training, optimistic): 0.9567
Note: only the OOF number (0.8132) reflects generalization.


## Stage 12 — Save bundle for the Kaggle inference notebook

Everything the inference notebook needs:

- `proto_ssm.pt`        — LightProtoSSM weights
- `residual_ssm.pt`     — ResidualSSM weights
- `architecture.json`   — model hyper-parameters (so you can rebuild without copy-pasting the class)
- `metadata.npz`        — species_cols, audio constants, site/hour mappings, correction_weight
- `perch_mapping.npz`   — direct + genus-proxy index tables
- `metrics.json`        — OOF metrics

In [15]:
# 1) Model weights
# Unwrap SWA AveragedModel wrapper if used — it prefixes all keys with "module."
_proto_state = (proto_final.module.state_dict()
                if hasattr(proto_final, 'module') else proto_final.state_dict())
torch.save(_proto_state,           AUTOSAVE_DIR / 'proto_ssm.pt')
torch.save(res_model.state_dict(), AUTOSAVE_DIR / 'residual_ssm.pt')

# 2) Architecture spec
arch_spec = {
    'proto_ssm': {
        'arch': 'light_proto_ssm',
        'd_input': int(EMB_DIM),
        'd_model': 128,
        'd_state': 16,
        'n_classes': int(n_species),
        'n_windows': int(N_WINDOWS),
        'dropout': 0.15,
        'n_sites': int(N_SITES_CAP),
        'meta_dim': 16,
        'use_cross_attn': True,
        'cross_attn_heads': 2,
    },
    'residual_ssm': {
        'arch': 'residual_ssm',
        'd_input': int(EMB_DIM),
        'd_scores': int(n_species),
        'd_model': 64,
        'd_state': 8,
        'n_classes': int(n_species),
        'n_windows': int(N_WINDOWS),
        'dropout': 0.1,
        'n_sites': int(N_SITES_CAP),
        'meta_dim': 8,
    },
    'inference': {
        'correction_weight': float(CORRECTION_WEIGHT),
        'tta_shifts': [0, 1, -1, 2, -2],
    },
}
(AUTOSAVE_DIR / 'architecture.json').write_text(json.dumps(arch_spec, indent=2))

# 3) Metadata + mappings
np.savez_compressed(
    AUTOSAVE_DIR / 'metadata.npz',
    species_cols   = np.array(species_cols, dtype=object),
    emb_dim        = np.int32(EMB_DIM),
    sr_load        = np.int32(SR_LOAD),
    perch_sr       = np.int32(PERCH_SR),
    window_sec     = np.float32(WINDOW_SEC),
    n_windows      = np.int32(N_WINDOWS),
    n_sites_cap    = np.int32(N_SITES_CAP),
    site_keys      = np.array(list(site2i.keys()), dtype=object),
    site_vals      = np.array(list(site2i.values()), dtype=np.int64),
    correction_weight = np.float32(CORRECTION_WEIGHT),
)

proxy_keys = np.array(list(proxy_map.keys()), dtype=np.int32)
proxy_vals = np.array([np.array(v, dtype=np.int32) for v in proxy_map.values()], dtype=object)
np.savez_compressed(
    AUTOSAVE_DIR / 'perch_mapping.npz',
    BC_INDICES    = BC_INDICES,
    MAPPED_POS    = MAPPED_POS,
    MAPPED_BC_IDX = MAPPED_BC_IDX,
    UNMAPPED_POS  = UNMAPPED_POS,
    proxy_keys    = proxy_keys,
    proxy_vals    = proxy_vals,
    NO_LABEL      = np.int32(NO_LABEL),
)

# 4) Metrics
(AUTOSAVE_DIR / 'metrics.json').write_text(json.dumps({
    'auc_perch_zero_shot':      float(auc_perch),
    'auc_protossm_oof':         float(auc_proto),
    'auc_protossm_residual_optimistic': float(auc_corrected),
    'fold_train_losses':        [float(x) for x in fold_losses],
    'n_files':                  int(n_files),
    'n_active_classes':         int(n_active),
}, indent=2))

print('Saved bundle to', AUTOSAVE_DIR)
for f in sorted(AUTOSAVE_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.1f} KB)')

Saved bundle to /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/protossm/protossm_run
  architecture.json  (0.6 KB)
  metadata.npz  (3.3 KB)
  metrics.json  (0.3 KB)
  perch_mapping.npz  (3.2 KB)
  proto_ssm.pt  (2916.5 KB)
  residual_ssm.pt  (716.0 KB)


## Stage 13 — Sanity check: round-trip the saved bundle

Quick reload of the saved weights so we know the inference notebook will work.

In [16]:
import json as _json
spec = _json.loads((AUTOSAVE_DIR / 'architecture.json').read_text())
ps = spec['proto_ssm']
rs = spec['residual_ssm']

m1 = LightProtoSSM(
    d_input=ps['d_input'], d_model=ps['d_model'], d_state=ps['d_state'],
    n_classes=ps['n_classes'], n_windows=ps['n_windows'], dropout=ps['dropout'],
    n_sites=ps['n_sites'], meta_dim=ps['meta_dim'],
    use_cross_attn=ps['use_cross_attn'], cross_attn_heads=ps['cross_attn_heads'],
)
m1.load_state_dict(torch.load(AUTOSAVE_DIR / 'proto_ssm.pt', map_location='cpu'))
m1.eval()

m2 = ResidualSSM(
    d_input=rs['d_input'], d_scores=rs['d_scores'], d_model=rs['d_model'],
    d_state=rs['d_state'], n_classes=rs['n_classes'], n_windows=rs['n_windows'],
    dropout=rs['dropout'], n_sites=rs['n_sites'], meta_dim=rs['meta_dim'],
)
m2.load_state_dict(torch.load(AUTOSAVE_DIR / 'residual_ssm.pt', map_location='cpu'))
m2.eval()

# Run a single forward to catch any shape bug
with torch.no_grad():
    e = torch.randn(1, N_WINDOWS, EMB_DIM)
    s = torch.randn(1, N_WINDOWS, n_species)
    si = torch.zeros(1, dtype=torch.long)
    hi = torch.zeros(1, dtype=torch.long)
    fp = m1(e, s, site_ids=si, hours=hi)
    rc = m2(e, fp, site_ids=si, hours=hi)
print(f'Round-trip OK. proto_out={tuple(fp.shape)}  residual_out={tuple(rc.shape)}')
print('\n✓ Bundle ready for the Kaggle inference notebook.')

Round-trip OK. proto_out=(1, 12, 234)  residual_out=(1, 12, 234)

✓ Bundle ready for the Kaggle inference notebook.
